In [1]:
from rich.console import Console, Group
from rich.panel import Panel


async def pprint(result, question):
    console = Console()
    answer_panel = Panel(result["messages"][-1].content, title="Answer")
    group = Group(*[
        Panel(msg.model_dump_json(indent=2), title=f"{i}: {type(msg).__name__}")
        for i, msg in enumerate(result["messages"])
    ])
    messages_pannel = Panel(group, title="Messages")
    full_panel = Panel(Group(answer_panel, messages_pannel), title=question)
    console.print(full_panel)


In [ ]:
import os

from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="deepseek-chat",
    temperature=0.7,
    model_provider="openai",
    api_key=os.environ.get("DEEPSEEK_API"),
    base_url="https://api.deepseek.com/v1",
)

from langchain.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient({
    "Math": {
        "transport": "stdio",
        "command": "uv",
        "args": ["run", "-m", "s5_mcp_servers_math", "start"],
    },
    "Weather": {"transport": "streamable-http", "url": "http://localhost:8000/mcp"},
})


tools = await mcp_client.get_tools()


console = Console()
console.print(Panel("\n".join([t.name for t in tools]), title="加载的工具"))

from langchain.agents import create_agent

agent = create_agent(model=llm, tools=tools, system_prompt="你是一个助理, 负责帮助计算和天气")

math_question = "请帮我计算 (3+5)*12 的结果"
math_result = await agent.ainvoke({"messages": [HumanMessage(content=math_question)]})
await pprint(math_result, math_question)

weather_question = "告诉我成都今天的天气"
weather_result = await agent.ainvoke({"messages": [HumanMessage(content=weather_question)]})
await pprint(weather_result, weather_question)


╭────────────────────────────────────────────────── 加载的工具 ───────────────────────────────────────────────────╮
│ add                                                                                                             │
│ multipy                                                                                                         │
│ geocde_city                                                                                                     │
│ get_current_weather                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────── 请帮我计算 (3+5)*12 的结果 ───────────────────────────────────────────╮
│ ╭────────────────────────────────────────────────── Answer ───────────────────────────────────────────────────╮ │
│ │ 计算结果为：**(3+5)×12 = 96** ✅                                                                            │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
│ ╭───────────────────────────────────────────────── Messages ──────────────────────────────────────────────────╮ │
│ │ ╭──────────────────────────────────────────── 0: HumanMessage ────────────────────────────────────────────╮ │ │
│ │ │ {                                                                                                       │ │ │
│ │ │   "content": "请帮我计算 (3+5)*12 的结果",                                                              │ │ │
│ │ │   "additional_kwargs": {},                                                                              │ │ │
│ │ │   "response_metadata": {},                                                                              │ │ │
│ │ │   "type": "human",                                                                                      │ │ │
│ │ │   "name": null,                                                                                         │ │ │
│ │ │   "id": "68dcab66-858b-4d04-bfd9-8a92a2dc859b"                                                          │ │ │
│ │ │ }                                                                                                       │ │ │
│ │ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │ │
│ │ ╭───────────────────────────────────────────── 1: AIMessage ──────────────────────────────────────────────╮ │ │
│ │ │ {                                                                                                       │ │ │
│ │ │   "content": "好的，我来计算 (3+5)*12。首先计算括号内的加法，然后乘以12。",                             │ │ │
│ │ │   "additional_kwargs": {                                                                                │ │ │
│ │ │     "refusal": null                                                                                     │ │ │
│ │ │   },                                                                                                    │ │ │
│ │ │   "response_metadata": {                                                                                │ │ │
│ │ │     "token_usage": {                                                                                    │ │ │
│ │ │       "completion_tokens": 79,                                                                          │ │ │
│ │ │       "prompt_tokens": 665,                                                                             │ │ │
│ │ │       "total_tokens": 744,                                                                              │ │ │
│ │ │       "completion_tokens_details": null,                                                                │ │ │
│ │ │       "prompt_tokens_details": {                                                                        │ │ │
│ │ │         "audio_tokens": null,                                                                           │ │ │
│ │ │         "cached_tokens": 640                                                                            │ │ │
│ │ │       },                                                                                                │ │ │
│ │ │       "prompt_cache_hit_tokens": 640,                                                                   │ │ │
│ │ │       "prompt_cache_miss_tokens": 25                                                                    │ │ │
│ │ │     },                                                                                                  │ │ │
│ │ │     "model_provider": "openai",                                                                  